# Media generation

### Imports

In [1]:
import pandas as pd
import os

### Directories and files to read in

In [82]:
media_dir = "/home/emma/Dokumente/thesis/media_creation/raw_csv/"
media_save_dir = "/home/emma/Dokumente/thesis/media_creation/classed_media"

In [90]:
translate2bigg = pd.read_csv("/home/emma/Dokumente/thesis/media_creation/translate_tobigg.csv", delimiter=",")

### Functions

In [91]:
def create_scaled_media(raw_data_csv):
    df_averaged = raw_data_csv.groupby("Compound", as_index=False)["Resp.Ratio"].mean()
    df_sorted = df_averaged.sort_values(by="Resp.Ratio", ascending=False) 
    compounds_rela = dict(zip(df_sorted["Compound"], df_sorted["Resp.Ratio"]))
    max_ab = max(compounds_rela.values())
    if max_ab == 0:
        return {comp_id: 0.1 for comp_id in compounds_rela}
    
    scale_fac = 1000 / max_ab
    
    compounds_scaled = {}
    for comp_id, abundance in compounds_rela.items():
        scaled_val = abundance * scale_fac
        compounds_scaled[comp_id] = scaled_val
        
    return compounds_scaled

In [ ]:
# Translate compound names to BiGG exchange reaction IDs and aggregate abundances per reaction.
def translate_to_bigg(scaled_dict, translate2bigg_df):
    mapping_dict = {
        str(name).strip(): str(reaction).strip()
        for name, reaction in zip(
            translate2bigg_df["Original Compound Name"],
            translate2bigg_df["BiGG Exchange Reaction"],
        )
        if pd.notna(name) and pd.notna(reaction)
    }

    translated_dict = {}
    for comp_name, abundance in scaled_dict.items():
        bigg_id = mapping_dict.get(str(comp_name).strip())
        if not bigg_id or bigg_id == "Exclude" or bigg_id == "nan":
            continue

        translated_dict[bigg_id] = translated_dict.get(bigg_id, 0.0) + abundance

    return translated_dict

In [93]:
def create_class_media(translated_dict):
    tier_media = {}
    for comp_id, scal_abund in translated_dict.items():
        if scal_abund <= 0.1:
            tier_media[comp_id] = 0.1
        elif 0.1 < scal_abund <= 10:
            tier_media[comp_id] = 10
        elif 10 < scal_abund <= 100:
            tier_media[comp_id] = 100
        elif 100 < scal_abund <= 500:
            tier_media[comp_id] = 500
        else: 
            tier_media[comp_id] = 1000 
            
    return tier_media

In [97]:
def save_media(medium_dict, medianame):
    media_df = pd.DataFrame.from_dict(
        medium_dict, orient="index")
    filename = f"{medianame}.csv"
    filepath = os.path.join(media_save_dir, filename)
    media_df.to_csv(filepath)    
    return media_df

In [98]:
def create_medium_total(raw_data_csv, translate2bigg, media_id):
    scaled = create_scaled_media(raw_data_csv)
    translated = translate_to_bigg(scaled, translate2bigg)
    classed = create_class_media(translated)
    save_media(classed, media_id)

### Main 

In [99]:
for file in os.listdir(media_dir):
    if file.endswith(".csv"):
        media_path = os.path.join(media_dir, file)
        raw_data = pd.read_csv(media_path, delimiter=",", thousands=",")
        media_id = os.path.splitext(file)[0]
        print(media_id)
        create_medium_total(raw_data, translate2bigg, media_id)

media_barley - 8dpi_AF_mock
media_barley - 7dpi_AF_mock
media_barley - 8dpi_Root_media_liquidphase
media_barley - 7dpi_Root_media_liquidphase
